# Exploration Notebook

Phase 1 scaffold notebook for ad-hoc data validation and EDA.

## Phase 2 validation

This section exercises the phase 2 science helpers against the processed parquet outputs.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.loader import get_latest_snapshot, get_list_size_ts
from science import (
    cluster_practices,
    flag_anomalies,
    flag_underserved,
    forecast_list_size,
    regional_inequality,
    size_imd_correlation,
)

latest_snapshot = get_latest_snapshot()
list_size_ts = get_list_size_ts()

print('latest_snapshot shape:', latest_snapshot.shape)
print('list_size_ts shape:', list_size_ts.shape)
print('latest snapshot columns:', latest_snapshot.columns.tolist())

latest_snapshot shape: (6145, 19)
list_size_ts shape: (501614, 4)
latest snapshot columns: ['SNAPSHOT_DATE', 'CODE', 'NUMBER_OF_PATIENTS', 'DATA_SOURCE', 'PRACTICE_CODE', 'PRACTICE_NAME', 'PCN_CODE', 'PCN_NAME', 'ICB_CODE', 'ICB_NAME', 'COMM_REGION_CODE', 'COMM_REGION_NAME', 'SUPPLIER_NAME', 'CLINICAL_SYSTEM', 'POSTCODE', 'IMD_SCORE', 'IMD_DECILE', 'LATITUDE', 'LONGITUDE']


In [3]:
national_ts = (
    list_size_ts.groupby('SNAPSHOT_DATE', as_index=False)['NUMBER_OF_PATIENTS']
    .sum()
    .sort_values('SNAPSHOT_DATE')
)
forecast = forecast_list_size(national_ts.tail(24), periods=3)

assert not forecast.empty
assert {'ds', 'yhat', 'yhat_lower', 'yhat_upper'}.issubset(forecast.columns)

forecast

00:00:42 - cmdstanpy - INFO - Chain [1] start processing
00:00:48 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
0,2026-07-01,6.304087e+07,6.304087e+07,6.304087e+07
1,2026-08-01,6.292561e+07,6.292561e+07,6.292561e+07
2,2026-09-01,6.276007e+07,6.272864e+07,6.277516e+07


In [4]:
anomaly_input = list_size_ts.head(2000).copy()
anomalies = flag_anomalies(anomaly_input)

assert {'MOM_CHANGE_ABS', 'MOM_CHANGE_PCT', 'ANOMALY_TYPE', 'ANOMALY_FLAG'}.issubset(anomalies.columns)
print('anomaly rows:', len(anomalies))
print('flagged rows:', int(anomalies['ANOMALY_FLAG'].sum()))
anomalies[['CODE', 'SNAPSHOT_DATE', 'ANOMALY_TYPE', 'ANOMALY_FLAG']].dropna(subset=['ANOMALY_TYPE']).head()

anomaly rows: 2000
flagged rows: 100


,CODE,SNAPSHOT_DATE,ANOMALY_TYPE,ANOMALY_FLAG


In [5]:
cluster_input = latest_snapshot.copy()
clustered = cluster_practices(cluster_input)

assert {'CLUSTER', 'CLUSTER_LABEL', 'UMAP_X', 'UMAP_Y', 'CLUSTER_SIZE'}.issubset(clustered.columns)
print('cluster count:', clustered['CLUSTER'].nunique())
clustered[['PRACTICE_CODE', 'CLUSTER', 'CLUSTER_SIZE', 'CLUSTER_DOMINANT_SYSTEM']].head()

/workspaces/nhs-gp-analytics/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


cluster count: 5


,PRACTICE_CODE,CLUSTER,CLUSTER_SIZE,CLUSTER_DOMINANT_SYSTEM
0,A81058,0,1299,EMIS Web
1,A81611,0,1299,EMIS Web
2,A82003,0,1299,EMIS Web
3,A82005,0,1299,EMIS Web
4,A82021,0,1299,EMIS Web


In [6]:
deprivation_input = latest_snapshot.copy()
underserved = flag_underserved(deprivation_input)
inequality = regional_inequality(deprivation_input)
correlation = size_imd_correlation(deprivation_input)

assert {'UNDER_SERVED', 'NATIONAL_MEDIAN_PATIENTS', 'DEPRIVED_AREA', 'SMALL_PRACTICE'}.issubset(underserved.columns)
assert {'GINI_COEFFICIENT', 'PRACTICE_COUNT', 'MEAN_PATIENTS', 'MEDIAN_PATIENTS'}.issubset(inequality.columns)
assert {'PEARSON_R', 'P_VALUE', 'N'}.issubset(correlation.columns)

print('underserved rows:', len(underserved))
print('inequality rows:', len(inequality))
print('correlation rows:', len(correlation))
underserved[underserved['UNDER_SERVED']].head()

underserved rows: 6145
inequality rows: 70
correlation rows: 7


,SNAPSHOT_DATE,CODE,NUMBER_OF_PATIENTS,DATA_SOURCE,PRACTICE_CODE,PRACTICE_NAME,PCN_CODE,PCN_NAME,ICB_CODE,ICB_NAME,...,CLINICAL_SYSTEM,POSTCODE,IMD_SCORE,IMD_DECILE,LATITUDE,LONGITUDE,NATIONAL_MEDIAN_PATIENTS,DEPRIVED_AREA,SMALL_PRACTICE,UNDER_SERVED
0,2026-06-01,A81001,3753,PDS,A81001,THE DENSHAM SURGERY,U89141,STOCKTON PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS181HU,77.084,1.0,54.561637,-1.318999,8814.0,True,True,True
3,2026-06-01,A81005,7540,PDS,A81005,SPRINGWOOD SURGERY,U07842,EAST CLEVELAND PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS147DJ,30.334,3.0,54.532610,-1.055459,8814.0,True,True,True
6,2026-06-01,A81009,7835,PDS,A81009,VILLAGE MEDICAL CENTRE,U85008,HOLGATE PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS56HF,57.687,1.0,54.562285,-1.241897,8814.0,True,True,True
8,2026-06-01,A81012,5617,PDS,A81012,WESTBOURNE MEDICAL CENTRE,U02671,GREATER MIDDLESBROUGH PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS36AL,65.896,1.0,54.571738,-1.216246,8814.0,True,True,True
10,2026-06-01,A81014,4097,PDS,A81014,QUEENSTREE PRACTICE,U94460,BILLINGHAM & NORTON PCN,QHM,NHS North East and North Cumbria Integrated Ca...,...,SystmOne,TS232LA,42.007,2.0,54.608303,-1.294813,8814.0,True,True,True


## Forecast backtesting and model comparison (DEC-004)

This section evaluates the production Prophet forecaster (`science/forecasting.py`)
against the baselines in `science/backtesting.py` — **naive** (repeat last value),
**seasonal naive** (repeat same month last year), and **linear** (the Prophet
fallback) — using rolling-origin backtesting. Methodology: `docs/FORECAST_VALIDATION.md`.

How to read the scores:

- **MASE** is the primary metric — mean absolute error scaled by the seasonal-naive
  in-sample error. Below 1.0 means the model beats seasonal naive; DEC-004 eliminates
  any model that doesn't.
- **coverage** is the fraction of actuals falling inside the forecast band. Prophet's
  default interval is 80%, so coverage well below 0.8 means the band is overconfident.
- Prefer the simplest model within ~2% MASE of the best.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from science.backtesting import (
    compare_models,
    default_forecasters,
    generate_cutoffs,
    rolling_origin_backtest,
    score_by_horizon,
)
from science.forecasting import Prophet

national_ts = (
    list_size_ts.groupby('SNAPSHOT_DATE', as_index=False)['NUMBER_OF_PATIENTS']
    .sum()
    .sort_values('SNAPSHOT_DATE')
    .reset_index(drop=True)
)

HORIZON, INITIAL, STEP = 12, 36, 6
if len(national_ts) < INITIAL + HORIZON:
    INITIAL = max(18, len(national_ts) - HORIZON)
    print(f'Short history ({len(national_ts)} months) - reduced initial training window to {INITIAL}.')

cutoffs = generate_cutoffs(
    national_ts.rename(columns={'SNAPSHOT_DATE': 'ds', 'NUMBER_OF_PATIENTS': 'y'}),
    horizon=HORIZON, initial=INITIAL, step=STEP,
)
print('months of history:', len(national_ts))
print('backtest cutoffs:', [cutoff.strftime('%Y-%m') for cutoff in cutoffs])
if Prophet is None:
    print('WARNING: prophet is not installed in this kernel - only the baselines will be scored.')
elif INITIAL < 24:
    print('NOTE: forecast_list_size falls back to its linear model when training history < 24 months.')

In [ ]:
# Every model is backtested on identical cutoffs, and each fit only sees data up to
# its cutoff. Prophet refits once per cutoff, so this cell can take a minute.
summary = compare_models(national_ts, horizon=HORIZON, initial=INITIAL, step=STEP)
summary.round(3)

In [ ]:
# How does error grow as the forecast horizon extends from 1 to 12 months ahead?
backtests = {
    name: rolling_origin_backtest(national_ts, forecaster, horizon=HORIZON, initial=INITIAL, step=STEP)
    for name, forecaster in default_forecasters().items()
}

fig, ax = plt.subplots(figsize=(9, 4))
for name, results in backtests.items():
    if results.empty:
        continue
    score_by_horizon(results)['mape'].mul(100).plot(ax=ax, marker='o', label=name)
ax.set_xlabel('months ahead')
ax.set_ylabel('MAPE (%)')
ax.set_title('Error growth by forecast horizon - national list size')
ax.legend()
plt.show()

In [ ]:
# Visual sanity check: the best-ranked model's forecast from the latest cutoff,
# with its uncertainty band, against what actually happened.
best_name = summary['mase'].idxmin()
results = backtests[best_name]
last_cutoff = results['cutoff'].max()
window = results[results['cutoff'] == last_cutoff]
history = national_ts[national_ts['SNAPSHOT_DATE'] <= last_cutoff].tail(36)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history['SNAPSHOT_DATE'], history['NUMBER_OF_PATIENTS'], label='history')
ax.plot(window['ds'], window['y'], color='black', label='actual')
ax.plot(window['ds'], window['yhat'], linestyle='--', label=f'{best_name} forecast')
ax.fill_between(window['ds'], window['yhat_lower'], window['yhat_upper'], alpha=0.2)
ax.axvline(last_cutoff, color='grey', linestyle=':', label='cutoff')
ax.set_title(f'{best_name}: {HORIZON}-month forecast from {last_cutoff:%Y-%m} vs actuals')
ax.legend()
plt.show()

In [ ]:
# Regional evaluation (DEC-004 protocol step 2): score every model per region, then
# aggregate as median and worst-case MASE. Uses the latest snapshot's practice-to-region
# mapping, so recently closed practices drop out - acceptable for exploration.
# Prophet refits per region per cutoff, so expect a few minutes' runtime.
region_map = (
    latest_snapshot[['PRACTICE_CODE', 'COMM_REGION_NAME']]
    .dropna()
    .drop_duplicates()
)
regional_ts = (
    list_size_ts.merge(region_map, left_on='CODE', right_on='PRACTICE_CODE', how='inner')
    .groupby(['COMM_REGION_NAME', 'SNAPSHOT_DATE'], as_index=False)['NUMBER_OF_PATIENTS']
    .sum()
)

regional_mase = pd.DataFrame(
    {
        region: compare_models(frame, horizon=HORIZON, initial=INITIAL, step=STEP)['mase']
        for region, frame in regional_ts.groupby('COMM_REGION_NAME')
    }
).T

pd.concat([regional_mase, regional_mase.agg(['median', 'max'])]).round(3)

### Why rolling-origin backtesting and not k-fold cross-validation?

Standard k-fold CV randomly shuffles observations into folds, which assumes they are
independent and exchangeable. A time series violates both assumptions, and applying
k-fold to it fails in two distinct ways:

1. **Temporal leakage.** Random folds train on future months to predict past ones.
   Because the series is trending and autocorrelated, the model effectively
   *interpolates* between known points either side of each test month instead of
   *extrapolating* into the unknown — scores come out flattering and say nothing about
   real forecast skill.
2. **Task mismatch.** Production asks exactly one question: "given history up to month
   *t*, what happens in months *t*+1 … *t*+12?" A random fold never poses that
   question, so even a leak-free variant would measure the wrong thing.

Rolling-origin backtesting *is* cross-validation adapted to time: each cutoff plays the
role of a fold, but every train/test split respects chronological order and every test
point is a genuine out-of-sample forecast at the horizon the dashboard serves. (Blocked
and purged k-fold variants for dependent data exist — common in finance — but the
standard tool for forecast model selection is rolling-origin evaluation; it is also
what `prophet.diagnostics.cross_validation` implements.)

Interpretation notes for the results above:

- Check MASE first: any model at or above 1.0 is losing to "same month last year".
- Check coverage against the nominal 80% level — a tight band that misses is worse
  than no band on the dashboard.
- Look at the horizon plot: a model that wins at 12 months but loses at 1-3 months
  (or vice versa) may justify different models for different dashboard views.
- Eyeball behaviour around the NHAIS-to-PDS transition (~early 2023): Prophet's
  changepoint handling is the main reason it was chosen, so it should visibly recover
  faster there than the baselines.